# Cross-Domain Evaluation of Retrieval-Augmented Generation (RAG)

**Course:** Advanced Topics in Artificial Intelligence  
**Team:** Aditya Jekkula · Ibaad Mohammed · Nate Dailey

---

## Overview

This notebook implements a controlled experimental study of Retrieval-Augmented Generation (RAG) across multiple knowledge domains. The goal is to understand how **domain mismatch** affects retrieval quality and answer generation — not just to build a RAG system, but to study *where and why it fails*.

### Datasets
| Dataset | Domain | Answer Type |
|---|---|---|
| SQuAD | General Wikipedia (structured passages) | Short extractive span |
| Natural Questions (NQ) | General Wikipedia (open search queries) | Short extractive answer |
| TriviaQA | Trivia / quiz (entertainment, history, science, sport) | Short extractive answer |

### Why TriviaQA instead of PubMedQA?
PubMedQA was replaced because its `long_answer` ground truths are author-written conclusions that do not appear verbatim in the source abstracts. This makes Recall@K always 0% regardless of retrieval quality, making quantitative evaluation impossible. TriviaQA provides short extractive answers directly comparable to SQuAD and NQ, while still offering strong domain contrast through its trivia/quiz source material.

### Evaluation Scenarios
| Scenario | Query Domain | Corpus Domain | Type |
|---|---|---|---|
| A | SQuAD | SQuAD | In-domain |
| B | NQ | NQ | In-domain |
| C | TriviaQA | TriviaQA | In-domain |
| D | SQuAD | NQ | Cross-domain |
| E | SQuAD | TriviaQA | Cross-domain |
| F | TriviaQA | SQuAD | Cross-domain |

### Components Evaluated Independently
1. **BM25 Retriever** — sparse keyword-based retrieval
2. **Dense Retriever** — semantic embedding-based retrieval (MiniLM + FAISS)
3. **LLM-Only Baseline** — generation with no retrieval (pure Qwen 2.5-7B)
4. **Full RAG** — retrieval + generation combined

### Key Research Questions
1. How does cross-domain retrieval affect retrieval relevance (Recall@K, MRR)?
2. How does domain mismatch impact answer quality (F1, EM)?
3. Where does the RAG pipeline fail — retrieval, generation, or both?
4. Does bad retrieval hurt generation more than no retrieval at all?

---
## Section 0 — Configuration

Set all experiment parameters here. Change `N_SAMPLES` to control evaluation size.
- `N_SAMPLES = 50` for a quick test run (~15 min on A100)
- `N_SAMPLES = 200` for the full experiment (~2 hrs on A100)

In [ ]:
# ── EXPERIMENT CONFIGURATION ──────────────────────────────────────

N_SAMPLES     = 200    # number of samples per dataset — set to 200 for full run
N_SPOT_CHECK  = 5     # qualitative examples printed per scenario
K_RETRIEVAL   = 5     # chunks retrieved for Recall@K
K_GENERATE    = 3     # chunks fed to generator
CHUNK_WORDS   = 250   # max words per chunk
CHUNK_OVERLAP = 50    # overlap between chunks
MODEL_NAME    = "Qwen/Qwen2.5-0.5B-Instruct"
EMBED_MODEL   = 'all-MiniLM-L6-v2'
PROJECT_PATH  = '/content/drive/MyDrive/RAG_Project'

print('Configuration set.')
print(f'  N_SAMPLES    = {N_SAMPLES}')
print(f'  K_RETRIEVAL  = {K_RETRIEVAL}')
print(f'  K_GENERATE   = {K_GENERATE}')
print(f'  MODEL        = {MODEL_NAME}')

Configuration set.
  N_SAMPLES    = 200
  K_RETRIEVAL  = 5
  K_GENERATE   = 3
  MODEL        = Qwen/Qwen2.5-0.5B-Instruct


---
## Section 1 — Environment Setup

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

In [ ]:
# import os
# os.makedirs(PROJECT_PATH, exist_ok=True)
# os.chdir(PROJECT_PATH)
# print('Working directory:', os.getcwd())

In [ ]:
!pip install -q transformers datasets sentence-transformers faiss-cpu rank_bm25 accelerate bitsandbytes nltk

In [ ]:
import re
import json
import time
import nltk
import torch
import numpy as np
import pandas as pd
from pathlib import Path
from nltk.tokenize import sent_tokenize
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer
import faiss

nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

print('All libraries imported successfully.')

All libraries imported successfully.


---
## Section 2 — Load Language Model

We use **Qwen 0.5-7B-Instruct** as the single generator across all experiments. Keeping the generator fixed ensures that performance differences are due to retrieval quality and domain mismatch — not the model itself.

In [ ]:
print(f'Loading tokenizer: {MODEL_NAME}')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print('Loading model (this may take a few minutes)...')
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map='auto',
    torch_dtype=torch.float16
)
print('Model loaded successfully.')
print(f'Device: {next(model.parameters()).device}')

Loading tokenizer: Qwen/Qwen2.5-0.5B-Instruct
Loading model (this may take a few minutes)...


Loading weights: 100%|██████████| 290/290 [00:01<00:00, 277.05it/s]


Model loaded successfully.
Device: cuda:0


In [ ]:
# Sanity check
def test_generation():
    prompt = 'Answer the question.\n\nQuestion: What is the capital of France?\nAnswer:'
    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
    outputs = model.generate(**inputs, max_new_tokens=20, do_sample=False)
    result = tokenizer.decode(outputs[0], skip_special_tokens=True)
    print('Sanity check:', result.split('Answer:')[-1].strip())

test_generation()

Sanity check: Paris
Explanation: The capital city of France is Paris. It is located in the south of the


---
## Section 3 — Load and Prepare Datasets

All three datasets are normalized into a common format: `{ question, context, answer }`.

Each dataset has two pools:
- **Eval pool** (`N_SAMPLES` items) — used for evaluation questions
- **Corpus pool** (larger separate set) — used to build the retrieval index

Keeping these separate ensures the retriever is not trivially memorizing the eval set.

### SQuAD
Stanford QA dataset. Questions target specific passages from Wikipedia articles. Answers are short extractive spans guaranteed to appear in the passage. Clean, well-structured, widely used as a baseline.

### Natural Questions (NQ)
Real Google search queries answered using Wikipedia. More natural and open-ended than SQuAD — users didn't have a passage in front of them when asking. Short extractive answers.

### TriviaQA (rc subset)
Trivia and quiz questions sourced from quiz leagues, trivia websites, and books. Topics span entertainment, history, science, sport, and geography — a genuinely different domain from Wikipedia-grounded SQuAD and NQ. Short extractive answers directly comparable to the other two datasets.

In [ ]:
# ── Load SQuAD ────────────────────────────────────────────────────
print('Loading SQuAD...')
squad_raw = load_dataset('squad', split='train')

def process_squad(example):
    return {
        'question': example['question'],
        'context':  example['context'],
        'answer':   example['answers']['text'][0] if example['answers']['text'] else ''
    }

squad_data = [process_squad(x) for x in squad_raw.select(range(N_SAMPLES * 3))]
squad_data = [x for x in squad_data if x['answer']][:N_SAMPLES]

squad_corpus_pool = [process_squad(x) for x in squad_raw.select(range(5000))]
squad_corpus_pool = [x for x in squad_corpus_pool if x['context']]

print(f'SQuAD eval samples:  {len(squad_data)}')
print(f'SQuAD corpus pool:   {len(squad_corpus_pool)}')
print(f'  Example Q: {squad_data[0]["question"]}')
print(f'  Example A: {squad_data[0]["answer"]}')

Loading SQuAD...
SQuAD eval samples:  200
SQuAD corpus pool:   5000
  Example Q: To whom did the Virgin Mary allegedly appear in 1858 in Lourdes France?
  Example A: Saint Bernadette Soubirous


In [ ]:
# ── Load Natural Questions ────────────────────────────────────────
print('Loading Natural Questions (streaming — may take 10-15 min)...')
nq_raw = load_dataset('google-research-datasets/natural_questions',
                       'default', split='train', streaming=True)

def process_nq(example):
    document_text = example.get('document', {}).get('html', '') or ''
    # Remove script and style blocks first
    context = re.sub(r'<(script|style)[^>]*>.*?</(script|style)>', '', document_text, flags=re.DOTALL)
    # Remove all remaining HTML tags
    context = re.sub(r'<[^>]+>', ' ', context)
    # Remove CSS blocks like .mw-spinner{...}
    context = re.sub(r'\{[^}]+\}', ' ', context)
    # Collapse all whitespace
    context = re.sub(r'\s+', ' ', context).strip()
    context = context[:2000]
    annotations = example.get('annotations', {})
    short_answers = annotations.get('short_answers', [])
    answer = ''
    if short_answers and len(short_answers) > 0:
        sa = short_answers[0]
        if isinstance(sa, dict) and sa.get('text'):
            answer = sa['text'][0] if isinstance(sa['text'], list) else sa['text']
    return {
        'question': example.get('question', {}).get('text', ''),
        'context':  context,
        'answer':   str(answer)
    }

nq_pool = []
for item in nq_raw:
    processed = process_nq(item)
    if processed['answer'] and processed['question'] and len(processed['context']) > 100:
        nq_pool.append(processed)
    if len(nq_pool) >= N_SAMPLES + 2000:
        break

nq_data        = nq_pool[:N_SAMPLES]
nq_corpus_pool = nq_pool[N_SAMPLES:]

print(f'NQ eval samples:  {len(nq_data)}')
print(f'NQ corpus pool:   {len(nq_corpus_pool)}')
print(f'  Example Q: {nq_data[0]["question"]}')
print(f'  Example A: {nq_data[0]["answer"]}')

Loading Natural Questions (streaming — may take 10-15 min)...
NQ eval samples:  200
NQ corpus pool:   2000
  Example Q: when is the last episode of season 8 of the walking dead
  Example A: March 18, 2018


In [ ]:
# ── Load TriviaQA ─────────────────────────────────────────────────
print('Loading TriviaQA (rc subset)...')
trivia_raw = load_dataset('trivia_qa', 'rc', split='train')

def process_trivia(example):
    # Use the first search result context as the passage
    search_results = example.get('search_results', {})
    contexts = search_results.get('search_context', [])
    context = contexts[0] if contexts else ''
    # Get the normalized answer value
    answer = example['answer'].get('value', '')
    return {
        'question': example['question'],
        'context':  context[:2000],
        'answer':   answer
    }

trivia_all = [process_trivia(x) for x in trivia_raw.select(range(min(N_SAMPLES * 3 + 3000, len(trivia_raw))))]
trivia_all = [x for x in trivia_all if x['answer'] and x['context'] and len(x['context']) > 100]

trivia_data        = trivia_all[:N_SAMPLES]
trivia_corpus_pool = trivia_all[N_SAMPLES:N_SAMPLES + 3000]

print(f'TriviaQA eval samples:  {len(trivia_data)}')
print(f'TriviaQA corpus pool:   {len(trivia_corpus_pool)}')
print(f'  Example Q: {trivia_data[0]["question"]}')
print(f'  Example A: {trivia_data[0]["answer"]}')

Loading TriviaQA (rc subset)...


Generating test split: 100%|██████████| 17210/17210 [00:07<00:00, 2448.31 examples/s]


TriviaQA eval samples:  200
TriviaQA corpus pool:   3000
  Example Q: Which American-born Sinclair won the Nobel Prize for Literature in 1930?
  Example A: Sinclair Lewis


---
## Section 4 — Text Chunking

Long contexts are split into overlapping sentence-aware chunks before indexing. This improves retrieval precision by targeting the most relevant passage rather than an entire document.

- **Max words per chunk:** `CHUNK_WORDS` (250)
- **Overlap:** `CHUNK_OVERLAP` (50 words) between consecutive chunks to avoid losing context at boundaries
- **Deduplication:** identical chunks removed to prevent BM25 returning the same chunk multiple times

In [ ]:
def chunk_text(text, max_words=CHUNK_WORDS, overlap=CHUNK_OVERLAP):
    """Split text into overlapping sentence-aware chunks."""
    sentences = sent_tokenize(text)
    chunks, current_chunk, current_len = [], [], 0
    for sentence in sentences:
        words = sentence.split()
        if current_len + len(words) > max_words and current_chunk:
            chunks.append(' '.join(current_chunk))
            overlap_words = ' '.join(current_chunk).split()[-overlap:]
            current_chunk = [' '.join(overlap_words)]
            current_len = len(overlap_words)
        current_chunk.append(sentence)
        current_len += len(words)
    if current_chunk:
        chunks.append(' '.join(current_chunk))
    return chunks


print('Building chunk corpora...')

squad_chunks = []
for item in squad_corpus_pool:
    squad_chunks.extend(chunk_text(item['context']))
squad_chunks = list(dict.fromkeys(squad_chunks))

nq_chunks = []
for item in nq_corpus_pool:
    nq_chunks.extend(chunk_text(item['context']))
nq_chunks = list(dict.fromkeys(nq_chunks))

trivia_chunks = []
for item in trivia_corpus_pool:
    trivia_chunks.extend(chunk_text(item['context']))
trivia_chunks = list(dict.fromkeys(trivia_chunks))

print(f'SQuAD chunks:    {len(squad_chunks)}')
print(f'NQ chunks:       {len(nq_chunks)}')
print(f'TriviaQA chunks: {len(trivia_chunks)}')

Building chunk corpora...
SQuAD chunks:    842
NQ chunks:       1983
TriviaQA chunks: 5409


---
## Section 5 — Build Retrievers

Two retrievers are built independently for each corpus:

**BM25 (Sparse):** Term-frequency ranking. Scores by keyword overlap weighted by document length. No semantic understanding — purely lexical.

**Dense (FAISS + MiniLM):** Encodes queries and chunks into 384-dim embedding vectors. Retrieves by L2 distance. Understands semantic meaning — synonyms and paraphrases can match without keyword overlap.

Both retrievers are fully independent — they share no components, enabling direct comparison.

In [ ]:
# ── BM25 Indexes ─────────────────────────────────────────────────
print('Building BM25 indexes...')
bm25_squad  = BM25Okapi([doc.split() for doc in squad_chunks])
bm25_nq     = BM25Okapi([doc.split() for doc in nq_chunks])
bm25_trivia = BM25Okapi([doc.split() for doc in trivia_chunks])
print('BM25 indexes ready.')

Building BM25 indexes...
BM25 indexes ready.


In [ ]:
# ── Dense Embedding Model ─────────────────────────────────────────
print(f'Loading embedding model: {EMBED_MODEL}')
embed_model = SentenceTransformer(EMBED_MODEL)
print('Embedding model ready.')

Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1821.70it/s]


Embedding model ready.


In [ ]:
# ── FAISS Indexes ─────────────────────────────────────────────────
print('Encoding and indexing chunks...')

def build_faiss_index(chunks):
    embeddings = embed_model.encode(chunks, convert_to_numpy=True, show_progress_bar=True)
    dim = embeddings.shape[1]
    index = faiss.IndexFlatL2(dim)
    index.add(embeddings)
    return index, embeddings

print('Encoding SQuAD chunks...')
faiss_squad, emb_squad = build_faiss_index(squad_chunks)

print('Encoding NQ chunks...')
faiss_nq, emb_nq = build_faiss_index(nq_chunks)

print('Encoding TriviaQA chunks...')
faiss_trivia, emb_trivia = build_faiss_index(trivia_chunks)

print('All FAISS indexes ready.')

Encoding and indexing chunks...
Encoding SQuAD chunks...


Batches: 100%|██████████| 27/27 [00:03<00:00,  7.26it/s]


Encoding NQ chunks...


Batches: 100%|██████████| 62/62 [00:11<00:00,  5.56it/s]


Encoding TriviaQA chunks...


Batches: 100%|██████████| 170/170 [00:28<00:00,  5.96it/s]

All FAISS indexes ready.


---
## Section 6 — Core Functions

All retrieval, generation, and evaluation functions are defined here. Each is independently callable so retrieval and generation can be analyzed in isolation.

In [ ]:
# ── Retrieval Functions ───────────────────────────────────────────

def retrieve_bm25(query, bm25_index, chunks, k=K_RETRIEVAL):
    """BM25 sparse retrieval with deduplication. Returns (text, score) list."""
    scores = bm25_index.get_scores(query.split())
    sorted_indices = np.argsort(scores)[::-1]
    results, seen = [], set()
    for idx in sorted_indices:
        text = chunks[idx]
        if text not in seen:
            results.append((text, float(scores[idx])))
            seen.add(text)
        if len(results) == k:
            break
    return results


def retrieve_dense(query, faiss_index, chunks, k=K_RETRIEVAL):
    """Dense semantic retrieval with deduplication. Returns (text, score) list."""
    q_emb = embed_model.encode([query], convert_to_numpy=True)
    D, I = faiss_index.search(q_emb, k * 3)
    results, seen = [], set()
    for dist, idx in zip(D[0], I[0]):
        if idx < len(chunks):
            text = chunks[idx]
            if text not in seen:
                results.append((text, float(dist)))
                seen.add(text)
        if len(results) == k:
            break
    return results


# ── Generation Functions ──────────────────────────────────────────

def build_rag_prompt(contexts, question):
    """RAG prompt — model must answer from context only."""
    context_text = '\n\n'.join(contexts)
    return (
        'Answer using the information in the context below. '
        'Give a concise, direct answer.\n\n'
        f'Context:\n{context_text}\n\n'
        f'Question: {question}\n\nAnswer:'
    )


def build_llm_only_prompt(question):
    """LLM-only baseline prompt — no context provided."""
    return (
        'Answer the following question as accurately and concisely as possible.\n\n'
        f'Question: {question}\n\nAnswer:'
    )


def generate_answer(prompt, max_new_tokens=60):
    """Generate answer using the loaded LLM. Cleans up leaked tokens."""
    inputs = tokenizer(prompt, return_tensors='pt',
                       truncation=True, max_length=2048).to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.eos_token_id,
        )
    result = tokenizer.decode(outputs[0], skip_special_tokens=True)
    answer = result.split('Answer:')[-1].strip()
    answer = answer.split('\n')[0].strip()
    answer = answer.split('Human:')[0].strip()
    answer = answer.split('Question:')[0].strip()
    answer = answer.split('You are an AI')[0].strip()
    answer = answer.split('User will')[0].strip()
    return answer.strip()


print('Retrieval and generation functions defined.')

Retrieval and generation functions defined.


In [ ]:
# ── Evaluation Metrics ────────────────────────────────────────────

def normalize(text):
    text = str(text).lower()
    text = re.sub(r'[^\w\s]', '', text)
    return text.strip()

def exact_match(pred, truth):
    return int(normalize(pred) == normalize(truth))

def f1_score(pred, truth):
    pred_tokens  = normalize(pred).split()
    truth_tokens = normalize(truth).split()
    common = set(pred_tokens) & set(truth_tokens)
    if not common:
        return 0.0
    precision = len(common) / len(pred_tokens)
    recall    = len(common) / len(truth_tokens)
    return 2 * precision * recall / (precision + recall)

def recall_at_k(answer, retrieved_chunks):
    """Token-overlap recall — 80% of answer tokens must appear in chunk."""
    ans_tokens = set(normalize(answer).split())
    if not ans_tokens:
        return 0
    for chunk in retrieved_chunks:
        chunk_tokens = set(normalize(chunk).split())
        overlap = ans_tokens & chunk_tokens
        if len(overlap) / len(ans_tokens) >= 0.8:
            return 1
    return 0

def mrr_score(answer, retrieved_chunks):
    """Mean Reciprocal Rank with token overlap."""
    ans_tokens = set(normalize(answer).split())
    if not ans_tokens:
        return 0.0
    for rank, chunk in enumerate(retrieved_chunks, start=1):
        chunk_tokens = set(normalize(chunk).split())
        overlap = ans_tokens & chunk_tokens
        if len(overlap) / len(ans_tokens) >= 0.8:
            return 1.0 / rank
    return 0.0

print('Evaluation metrics defined.')

Evaluation metrics defined.


---
## Section 7 — BM25 Retriever Evaluation

BM25 is evaluated **independently** — purely as a retriever before any generation. This tells us how well keyword-based retrieval finds relevant chunks across all scenarios.

**Metrics:**
- **Recall@K** — does the correct answer appear in the top-K retrieved chunks?
- **MRR** — Mean Reciprocal Rank — how high up is the first correct chunk?

In [ ]:
def evaluate_retriever(data, retriever_fn, label, n=N_SAMPLES, n_spot=N_SPOT_CHECK):
    """Evaluate a retriever independently. Returns metrics and spot-check examples."""
    recalls, mrrs = [], []
    spot_checks = []

    for i, sample in enumerate(data[:n]):
        q = sample['question']
        a = sample['answer']

        results = retriever_fn(q)
        chunks  = [r[0] for r in results]
        scores  = [r[1] for r in results]

        r   = recall_at_k(a, chunks)
        mrr = mrr_score(a, chunks)
        recalls.append(r)
        mrrs.append(mrr)

        if len(spot_checks) < n_spot:
            spot_checks.append({
                'question':   q,
                'answer':     a,
                'top_chunk':  chunks[0] if chunks else '',
                'top_score':  round(scores[0], 3) if scores else 0,
                'recall_hit': bool(r),
                'all_chunks': chunks
            })

        if (i+1) % 50 == 0:
            print(f'  [{label}] Processed {i+1}/{n}')

    return {
        'label':    label,
        'recall@k': round(np.mean(recalls), 4),
        'mrr':      round(np.mean(mrrs), 4),
        'n':        n
    }, spot_checks


def print_retrieval_spotcheck(spots, label, n=N_SPOT_CHECK):
    print(f'--- {label} ---')
    for i, ex in enumerate(spots[:n]):
        hit = '✅ HIT' if ex['recall_hit'] else '❌ MISS'
        print(f'\nExample {i+1} [{hit}]')
        print(f'  Question:  {ex["question"][:120]}')
        print(f'  Answer:    {ex["answer"][:100]}')
        print(f'  Top chunk: {ex["top_chunk"][:150]}...')
        print(f'  Score:     {ex["top_score"]}')
    print()

print('Retriever evaluation functions defined.')

Retriever evaluation functions defined.


In [ ]:
print('=== BM25 RETRIEVER EVALUATION ===')
bm25_results = {}

print('\nScenario A: SQuAD -> SQuAD (in-domain)...')
bm25_results['A_squad_squad'], bm25_spots_A = evaluate_retriever(
    squad_data, lambda q: retrieve_bm25(q, bm25_squad, squad_chunks), 'BM25 | SQuAD->SQuAD')

print('\nScenario B: NQ -> NQ (in-domain)...')
bm25_results['B_nq_nq'], bm25_spots_B = evaluate_retriever(
    nq_data, lambda q: retrieve_bm25(q, bm25_nq, nq_chunks), 'BM25 | NQ->NQ')

print('\nScenario C: TriviaQA -> TriviaQA (in-domain)...')
bm25_results['C_trivia_trivia'], bm25_spots_C = evaluate_retriever(
    trivia_data, lambda q: retrieve_bm25(q, bm25_trivia, trivia_chunks), 'BM25 | Trivia->Trivia')

print('\nScenario D: SQuAD -> NQ (cross-domain)...')
bm25_results['D_squad_nq'], bm25_spots_D = evaluate_retriever(
    squad_data, lambda q: retrieve_bm25(q, bm25_nq, nq_chunks), 'BM25 | SQuAD->NQ')

print('\nScenario E: SQuAD -> TriviaQA (cross-domain)...')
bm25_results['E_squad_trivia'], bm25_spots_E = evaluate_retriever(
    squad_data, lambda q: retrieve_bm25(q, bm25_trivia, trivia_chunks), 'BM25 | SQuAD->Trivia')

print('\nScenario F: TriviaQA -> SQuAD (cross-domain)...')
bm25_results['F_trivia_squad'], bm25_spots_F = evaluate_retriever(
    trivia_data, lambda q: retrieve_bm25(q, bm25_squad, squad_chunks), 'BM25 | Trivia->SQuAD')

print('\n=== BM25 RETRIEVAL RESULTS ===')
print(pd.DataFrame(bm25_results).T[['label','recall@k','mrr','n']].to_string())

=== BM25 RETRIEVER EVALUATION ===

Scenario A: SQuAD -> SQuAD (in-domain)...
  [BM25 | SQuAD->SQuAD] Processed 50/200
  [BM25 | SQuAD->SQuAD] Processed 100/200
  [BM25 | SQuAD->SQuAD] Processed 150/200
  [BM25 | SQuAD->SQuAD] Processed 200/200

Scenario B: NQ -> NQ (in-domain)...
  [BM25 | NQ->NQ] Processed 50/200
  [BM25 | NQ->NQ] Processed 100/200
  [BM25 | NQ->NQ] Processed 150/200
  [BM25 | NQ->NQ] Processed 200/200

Scenario C: TriviaQA -> TriviaQA (in-domain)...
  [BM25 | Trivia->Trivia] Processed 50/200
  [BM25 | Trivia->Trivia] Processed 100/200
  [BM25 | Trivia->Trivia] Processed 150/200
  [BM25 | Trivia->Trivia] Processed 200/200

Scenario D: SQuAD -> NQ (cross-domain)...
  [BM25 | SQuAD->NQ] Processed 50/200
  [BM25 | SQuAD->NQ] Processed 100/200
  [BM25 | SQuAD->NQ] Processed 150/200
  [BM25 | SQuAD->NQ] Processed 200/200

Scenario E: SQuAD -> TriviaQA (cross-domain)...
  [BM25 | SQuAD->Trivia] Processed 50/200
  [BM25 | SQuAD->Trivia] Processed 100/200
  [BM25 | SQuAD->Tri

In [ ]:
# ── BM25 Qualitative Spot-Check ───────────────────────────────────
print('=== BM25 QUALITATIVE SPOT-CHECK ===')
print('Showing 5 examples per scenario — retrieval hits and misses.\n')

print_retrieval_spotcheck(bm25_spots_A, 'BM25 | SQuAD->SQuAD (in-domain)')
print_retrieval_spotcheck(bm25_spots_B, 'BM25 | NQ->NQ (in-domain)')
print_retrieval_spotcheck(bm25_spots_C, 'BM25 | TriviaQA->TriviaQA (in-domain)')
print_retrieval_spotcheck(bm25_spots_E, 'BM25 | SQuAD->TriviaQA (cross-domain)')
print_retrieval_spotcheck(bm25_spots_F, 'BM25 | TriviaQA->SQuAD (cross-domain)')

=== BM25 QUALITATIVE SPOT-CHECK ===
Showing 5 examples per scenario — retrieval hits and misses.

--- BM25 | SQuAD->SQuAD (in-domain) ---

Example 1 [✅ HIT]
  Question:  To whom did the Virgin Mary allegedly appear in 1858 in Lourdes France?
  Answer:    Saint Bernadette Soubirous
  Top chunk: Architecturally, the school has a Catholic character. Atop the Main Building's gold dome is a golden statue of the Virgin Mary. Immediately in front o...
  Score:     20.586

Example 2 [✅ HIT]
  Question:  What is in front of the Notre Dame Main Building?
  Answer:    a copper statue of Christ
  Top chunk: The "Notre Dame Victory March" is the fight song for the University of Notre Dame. It was written by two brothers who were Notre Dame graduates. The R...
  Score:     25.393

Example 3 [✅ HIT]
  Question:  The Basilica of the Sacred heart at Notre Dame is beside to which structure?
  Answer:    the Main Building
  Top chunk: Because of its Catholic identity, a number of religious buildings stan

---
## Section 8 — Dense Retriever Evaluation

Dense retrieval is evaluated with the same framework as BM25, enabling direct comparison. The key question: does semantic understanding help in cross-domain settings, or does the domain gap defeat both retrievers equally?

In [ ]:
print('=== DENSE RETRIEVER EVALUATION ===')
dense_results = {}

print('\nScenario A: SQuAD -> SQuAD (in-domain)...')
dense_results['A_squad_squad'], dense_spots_A = evaluate_retriever(
    squad_data, lambda q: retrieve_dense(q, faiss_squad, squad_chunks), 'Dense | SQuAD->SQuAD')

print('\nScenario B: NQ -> NQ (in-domain)...')
dense_results['B_nq_nq'], dense_spots_B = evaluate_retriever(
    nq_data, lambda q: retrieve_dense(q, faiss_nq, nq_chunks), 'Dense | NQ->NQ')

print('\nScenario C: TriviaQA -> TriviaQA (in-domain)...')
dense_results['C_trivia_trivia'], dense_spots_C = evaluate_retriever(
    trivia_data, lambda q: retrieve_dense(q, faiss_trivia, trivia_chunks), 'Dense | Trivia->Trivia')

print('\nScenario D: SQuAD -> NQ (cross-domain)...')
dense_results['D_squad_nq'], dense_spots_D = evaluate_retriever(
    squad_data, lambda q: retrieve_dense(q, faiss_nq, nq_chunks), 'Dense | SQuAD->NQ')

print('\nScenario E: SQuAD -> TriviaQA (cross-domain)...')
dense_results['E_squad_trivia'], dense_spots_E = evaluate_retriever(
    squad_data, lambda q: retrieve_dense(q, faiss_trivia, trivia_chunks), 'Dense | SQuAD->Trivia')

print('\nScenario F: TriviaQA -> SQuAD (cross-domain)...')
dense_results['F_trivia_squad'], dense_spots_F = evaluate_retriever(
    trivia_data, lambda q: retrieve_dense(q, faiss_squad, squad_chunks), 'Dense | Trivia->SQuAD')

print('\n=== DENSE RETRIEVAL RESULTS ===')
print(pd.DataFrame(dense_results).T[['label','recall@k','mrr','n']].to_string())

=== DENSE RETRIEVER EVALUATION ===

Scenario A: SQuAD -> SQuAD (in-domain)...
  [Dense | SQuAD->SQuAD] Processed 50/200
  [Dense | SQuAD->SQuAD] Processed 100/200
  [Dense | SQuAD->SQuAD] Processed 150/200
  [Dense | SQuAD->SQuAD] Processed 200/200

Scenario B: NQ -> NQ (in-domain)...
  [Dense | NQ->NQ] Processed 50/200
  [Dense | NQ->NQ] Processed 100/200
  [Dense | NQ->NQ] Processed 150/200
  [Dense | NQ->NQ] Processed 200/200

Scenario C: TriviaQA -> TriviaQA (in-domain)...
  [Dense | Trivia->Trivia] Processed 50/200
  [Dense | Trivia->Trivia] Processed 100/200
  [Dense | Trivia->Trivia] Processed 150/200
  [Dense | Trivia->Trivia] Processed 200/200

Scenario D: SQuAD -> NQ (cross-domain)...
  [Dense | SQuAD->NQ] Processed 50/200
  [Dense | SQuAD->NQ] Processed 100/200
  [Dense | SQuAD->NQ] Processed 150/200
  [Dense | SQuAD->NQ] Processed 200/200

Scenario E: SQuAD -> TriviaQA (cross-domain)...
  [Dense | SQuAD->Trivia] Processed 50/200
  [Dense | SQuAD->Trivia] Processed 100/200
 

In [ ]:
# ── Dense Qualitative Spot-Check ──────────────────────────────────
print('=== DENSE RETRIEVER QUALITATIVE SPOT-CHECK ===')
print('Showing 5 examples per scenario.\n')

print_retrieval_spotcheck(dense_spots_A, 'Dense | SQuAD->SQuAD (in-domain)')
print_retrieval_spotcheck(dense_spots_B, 'Dense | NQ->NQ (in-domain)')
print_retrieval_spotcheck(dense_spots_C, 'Dense | TriviaQA->TriviaQA (in-domain)')
print_retrieval_spotcheck(dense_spots_E, 'Dense | SQuAD->TriviaQA (cross-domain)')
print_retrieval_spotcheck(dense_spots_F, 'Dense | TriviaQA->SQuAD (cross-domain)')

=== DENSE RETRIEVER QUALITATIVE SPOT-CHECK ===
Showing 5 examples per scenario.

--- Dense | SQuAD->SQuAD (in-domain) ---

Example 1 [✅ HIT]
  Question:  To whom did the Virgin Mary allegedly appear in 1858 in Lourdes France?
  Answer:    Saint Bernadette Soubirous
  Top chunk: Architecturally, the school has a Catholic character. Atop the Main Building's gold dome is a golden statue of the Virgin Mary. Immediately in front o...
  Score:     0.797

Example 2 [✅ HIT]
  Question:  What is in front of the Notre Dame Main Building?
  Answer:    a copper statue of Christ
  Top chunk: The University of Notre Dame du Lac (or simply Notre Dame /ˌnoʊtərˈdeɪm/ NOH-tər-DAYM) is a Catholic research university located adjacent to South Ben...
  Score:     0.816

Example 3 [✅ HIT]
  Question:  The Basilica of the Sacred heart at Notre Dame is beside to which structure?
  Answer:    the Main Building
  Top chunk: Because of its Catholic identity, a number of religious buildings stand on campus. The O

---
## Section 9 — BM25 vs Dense Retriever Comparison

Direct side-by-side comparison across all scenarios. This reveals whether semantic retrieval outperforms keyword retrieval, and whether the advantage holds under domain mismatch.

In [ ]:
print('=== BM25 vs DENSE: SIDE-BY-SIDE COMPARISON ===')

scenario_keys   = ['A_squad_squad','B_nq_nq','C_trivia_trivia','D_squad_nq','E_squad_trivia','F_trivia_squad']
scenario_labels = [
    'A: SQuAD->SQuAD (in-domain)',
    'B: NQ->NQ (in-domain)',
    'C: TriviaQA->TriviaQA (in-domain)',
    'D: SQuAD->NQ (cross-domain)',
    'E: SQuAD->TriviaQA (cross-domain)',
    'F: TriviaQA->SQuAD (cross-domain)'
]

comparison_rows = []
for key, label in zip(scenario_keys, scenario_labels):
    b = bm25_results.get(key, {})
    d = dense_results.get(key, {})
    comparison_rows.append({
        'Scenario':       label,
        'BM25 Recall@K':  b.get('recall@k', '-'),
        'Dense Recall@K': d.get('recall@k', '-'),
        'BM25 MRR':       b.get('mrr', '-'),
        'Dense MRR':      d.get('mrr', '-'),
        'Winner':         'Dense' if d.get('recall@k', 0) > b.get('recall@k', 0) else 'BM25'
    })

print(pd.DataFrame(comparison_rows).to_string(index=False))

=== BM25 vs DENSE: SIDE-BY-SIDE COMPARISON ===
                         Scenario  BM25 Recall@K  Dense Recall@K  BM25 MRR  Dense MRR Winner
      A: SQuAD->SQuAD (in-domain)          0.750           0.925    0.5838     0.7924  Dense
            B: NQ->NQ (in-domain)          0.015           0.015    0.0048     0.0075   BM25
C: TriviaQA->TriviaQA (in-domain)          0.255           0.325    0.1720     0.2458  Dense
      D: SQuAD->NQ (cross-domain)          0.015           0.005    0.0092     0.0012   BM25
E: SQuAD->TriviaQA (cross-domain)          0.055           0.050    0.0310     0.0209   BM25
F: TriviaQA->SQuAD (cross-domain)          0.025           0.070    0.0192     0.0435  Dense


---
## Section 10 — LLM-Only Baseline Evaluation

The LLM answers questions with **no retrieved context**. This baseline is essential for the project:
1. If LLM-only ≈ RAG, retrieval is not contributing
2. If cross-domain RAG < LLM-only, bad retrieval is actively hurting generation
3. It isolates what the model already knows from what retrieval adds

All three datasets are evaluated independently.

In [ ]:
def evaluate_llm_only(data, label, n=N_SAMPLES, n_spot=N_SPOT_CHECK):
    """Evaluate LLM-only baseline — no retrieval."""
    ems, f1s = [], []
    spot_checks = []

    for i, sample in enumerate(data[:n]):
        q = sample['question']
        a = sample['answer']

        pred = generate_answer(build_llm_only_prompt(q))
        em   = exact_match(pred, a)
        f1   = f1_score(pred, a)
        ems.append(em)
        f1s.append(f1)

        if len(spot_checks) < n_spot:
            spot_checks.append({
                'question':   q,
                'answer':     a,
                'prediction': pred,
                'em':         em,
                'f1':         round(f1, 3)
            })

        if (i+1) % 50 == 0:
            print(f'  [{label}] Processed {i+1}/{n}')

    return {
        'label': label,
        'em':    round(np.mean(ems), 4),
        'f1':    round(np.mean(f1s), 4),
        'n':     n
    }, spot_checks


def print_generation_spotcheck(spots, label, n=N_SPOT_CHECK):
    print(f'--- {label} ---')
    for i, ex in enumerate(spots[:n]):
        print(f'\nExample {i+1}')
        print(f'  Question:     {ex["question"][:120]}')
        print(f'  Ground truth: {ex["answer"][:120]}')
        print(f'  Prediction:   {ex["prediction"][:120]}')
        print(f'  EM: {ex["em"]} | F1: {ex["f1"]}')
    print()

print('LLM-only evaluation functions defined.')

LLM-only evaluation functions defined.


In [ ]:
print('=== LLM-ONLY BASELINE EVALUATION ===')
llm_results = {}

print('\nLLM-only on SQuAD...')
llm_results['squad'], llm_spots_squad = evaluate_llm_only(squad_data, 'LLM-Only | SQuAD')

print('\nLLM-only on Natural Questions...')
llm_results['nq'], llm_spots_nq = evaluate_llm_only(nq_data, 'LLM-Only | NQ')

print('\nLLM-only on TriviaQA...')
llm_results['trivia'], llm_spots_trivia = evaluate_llm_only(trivia_data, 'LLM-Only | TriviaQA')

print('\n=== LLM-ONLY BASELINE RESULTS ===')
print(pd.DataFrame(llm_results).T[['label','em','f1','n']].to_string())

=== LLM-ONLY BASELINE EVALUATION ===

LLM-only on SQuAD...
  [LLM-Only | SQuAD] Processed 50/200
  [LLM-Only | SQuAD] Processed 100/200
  [LLM-Only | SQuAD] Processed 150/200
  [LLM-Only | SQuAD] Processed 200/200

LLM-only on Natural Questions...
  [LLM-Only | NQ] Processed 50/200
  [LLM-Only | NQ] Processed 100/200
  [LLM-Only | NQ] Processed 150/200
  [LLM-Only | NQ] Processed 200/200

LLM-only on TriviaQA...
  [LLM-Only | TriviaQA] Processed 50/200
  [LLM-Only | TriviaQA] Processed 100/200
  [LLM-Only | TriviaQA] Processed 150/200
  [LLM-Only | TriviaQA] Processed 200/200

=== LLM-ONLY BASELINE RESULTS ===
                      label    em      f1    n
squad      LLM-Only | SQuAD   0.0  0.0278  200
nq            LLM-Only | NQ   0.0  0.0404  200
trivia  LLM-Only | TriviaQA  0.01  0.0401  200


In [ ]:
# ── LLM-Only Qualitative Spot-Check ──────────────────────────────
print('=== LLM-ONLY QUALITATIVE SPOT-CHECK ===')
print('Showing 5 examples per dataset — what does the model know without retrieval?\n')

print_generation_spotcheck(llm_spots_squad,  'LLM-Only | SQuAD')
print_generation_spotcheck(llm_spots_nq,     'LLM-Only | NQ')
print_generation_spotcheck(llm_spots_trivia, 'LLM-Only | TriviaQA')

=== LLM-ONLY QUALITATIVE SPOT-CHECK ===
Showing 5 examples per dataset — what does the model know without retrieval?

--- LLM-Only | SQuAD ---

Example 1
  Question:     To whom did the Virgin Mary allegedly appear in 1858 in Lourdes France?
  Ground truth: Saint Bernadette Soubirous
  Prediction:   The Virgin Mary allegedly appeared to a group of people in Lourdes, France in 1858. This event is known as the "Lourdes 
  EM: 0 | F1: 0.0

Example 2
  Question:     What is in front of the Notre Dame Main Building?
  Ground truth: a copper statue of Christ
  Prediction:   The Notre Dame Main Building, also known as the Notre Dame Cathedral or simply Notre Dame, is a massive Gothic cathedral
  EM: 0 | F1: 0.073

Example 3
  Question:     The Basilica of the Sacred heart at Notre Dame is beside to which structure?
  Ground truth: the Main Building
  Prediction:   The Basilica of the Sacred Heart at Notre Dame is located next to the Notre Dame Cathedral, a famous Gothic cathedral in
  EM: 0 |

---
## Section 11 — Full RAG Evaluation

The complete RAG pipeline: retrieved chunks are fed as context to the generator. We run this for both BM25 and Dense retrievers across all six scenarios.

Key question: **does retrieval help generation, and by how much does cross-domain retrieval hurt it?**

In [ ]:
def evaluate_rag(data, retriever_fn, label, n=N_SAMPLES, n_spot=N_SPOT_CHECK):
    """Evaluate full RAG pipeline — retrieval + generation."""
    recalls, ems, f1s = [], [], []
    spot_checks = []

    for i, sample in enumerate(data[:n]):
        q = sample['question']
        a = sample['answer']

        results   = retriever_fn(q)
        chunks    = [r[0] for r in results]
        scores    = [r[1] for r in results]
        top_gen   = chunks[:K_GENERATE]

        r    = recall_at_k(a, chunks)
        pred = generate_answer(build_rag_prompt(top_gen, q))
        em   = exact_match(pred, a)
        f1   = f1_score(pred, a)

        recalls.append(r)
        ems.append(em)
        f1s.append(f1)

        if len(spot_checks) < n_spot:
            spot_checks.append({
                'question':   q,
                'answer':     a,
                'top_chunk':  chunks[0] if chunks else '',
                'top_score':  round(scores[0], 3) if scores else 0,
                'prediction': pred,
                'recall_hit': bool(r),
                'em':         em,
                'f1':         round(f1, 3)
            })

        if (i+1) % 50 == 0:
            print(f'  [{label}] Processed {i+1}/{n}')

    return {
        'label':    label,
        'recall@k': round(np.mean(recalls), 4),
        'em':       round(np.mean(ems), 4),
        'f1':       round(np.mean(f1s), 4),
        'n':        n
    }, spot_checks

print('RAG evaluation function defined.')

RAG evaluation function defined.


In [ ]:
print('=== BM25 RAG EVALUATION ===')
rag_bm25 = {}

print('\nA: SQuAD -> SQuAD...')
rag_bm25['A'], rag_bm25_spots_A = evaluate_rag(
    squad_data, lambda q: retrieve_bm25(q, bm25_squad, squad_chunks), 'BM25-RAG | SQuAD->SQuAD')

print('\nB: NQ -> NQ...')
rag_bm25['B'], rag_bm25_spots_B = evaluate_rag(
    nq_data, lambda q: retrieve_bm25(q, bm25_nq, nq_chunks), 'BM25-RAG | NQ->NQ')

print('\nC: TriviaQA -> TriviaQA...')
rag_bm25['C'], rag_bm25_spots_C = evaluate_rag(
    trivia_data, lambda q: retrieve_bm25(q, bm25_trivia, trivia_chunks), 'BM25-RAG | Trivia->Trivia')

print('\nD: SQuAD -> NQ...')
rag_bm25['D'], rag_bm25_spots_D = evaluate_rag(
    squad_data, lambda q: retrieve_bm25(q, bm25_nq, nq_chunks), 'BM25-RAG | SQuAD->NQ')

print('\nE: SQuAD -> TriviaQA...')
rag_bm25['E'], rag_bm25_spots_E = evaluate_rag(
    squad_data, lambda q: retrieve_bm25(q, bm25_trivia, trivia_chunks), 'BM25-RAG | SQuAD->Trivia')

print('\nF: TriviaQA -> SQuAD...')
rag_bm25['F'], rag_bm25_spots_F = evaluate_rag(
    trivia_data, lambda q: retrieve_bm25(q, bm25_squad, squad_chunks), 'BM25-RAG | Trivia->SQuAD')

print('\n=== BM25 RAG RESULTS ===')
print(pd.DataFrame(rag_bm25).T[['label','recall@k','em','f1','n']].to_string())

=== BM25 RAG EVALUATION ===

A: SQuAD -> SQuAD...
  [BM25-RAG | SQuAD->SQuAD] Processed 50/200
  [BM25-RAG | SQuAD->SQuAD] Processed 100/200
  [BM25-RAG | SQuAD->SQuAD] Processed 150/200
  [BM25-RAG | SQuAD->SQuAD] Processed 200/200

B: NQ -> NQ...
  [BM25-RAG | NQ->NQ] Processed 50/200
  [BM25-RAG | NQ->NQ] Processed 100/200
  [BM25-RAG | NQ->NQ] Processed 150/200
  [BM25-RAG | NQ->NQ] Processed 200/200

C: TriviaQA -> TriviaQA...
  [BM25-RAG | Trivia->Trivia] Processed 50/200
  [BM25-RAG | Trivia->Trivia] Processed 100/200
  [BM25-RAG | Trivia->Trivia] Processed 150/200
  [BM25-RAG | Trivia->Trivia] Processed 200/200

D: SQuAD -> NQ...
  [BM25-RAG | SQuAD->NQ] Processed 50/200
  [BM25-RAG | SQuAD->NQ] Processed 100/200
  [BM25-RAG | SQuAD->NQ] Processed 150/200
  [BM25-RAG | SQuAD->NQ] Processed 200/200

E: SQuAD -> TriviaQA...
  [BM25-RAG | SQuAD->Trivia] Processed 50/200
  [BM25-RAG | SQuAD->Trivia] Processed 100/200
  [BM25-RAG | SQuAD->Trivia] Processed 150/200
  [BM25-RAG | SQuA

In [ ]:
print('=== DENSE RAG EVALUATION ===')
rag_dense = {}

print('\nA: SQuAD -> SQuAD...')
rag_dense['A'], rag_dense_spots_A = evaluate_rag(
    squad_data, lambda q: retrieve_dense(q, faiss_squad, squad_chunks), 'Dense-RAG | SQuAD->SQuAD')

print('\nB: NQ -> NQ...')
rag_dense['B'], rag_dense_spots_B = evaluate_rag(
    nq_data, lambda q: retrieve_dense(q, faiss_nq, nq_chunks), 'Dense-RAG | NQ->NQ')

print('\nC: TriviaQA -> TriviaQA...')
rag_dense['C'], rag_dense_spots_C = evaluate_rag(
    trivia_data, lambda q: retrieve_dense(q, faiss_trivia, trivia_chunks), 'Dense-RAG | Trivia->Trivia')

print('\nD: SQuAD -> NQ...')
rag_dense['D'], rag_dense_spots_D = evaluate_rag(
    squad_data, lambda q: retrieve_dense(q, faiss_nq, nq_chunks), 'Dense-RAG | SQuAD->NQ')

print('\nE: SQuAD -> TriviaQA...')
rag_dense['E'], rag_dense_spots_E = evaluate_rag(
    squad_data, lambda q: retrieve_dense(q, faiss_trivia, trivia_chunks), 'Dense-RAG | SQuAD->Trivia')

print('\nF: TriviaQA -> SQuAD...')
rag_dense['F'], rag_dense_spots_F = evaluate_rag(
    trivia_data, lambda q: retrieve_dense(q, faiss_squad, squad_chunks), 'Dense-RAG | Trivia->SQuAD')

print('\n=== DENSE RAG RESULTS ===')
print(pd.DataFrame(rag_dense).T[['label','recall@k','em','f1','n']].to_string())

=== DENSE RAG EVALUATION ===

A: SQuAD -> SQuAD...
  [Dense-RAG | SQuAD->SQuAD] Processed 50/200
  [Dense-RAG | SQuAD->SQuAD] Processed 100/200
  [Dense-RAG | SQuAD->SQuAD] Processed 150/200
  [Dense-RAG | SQuAD->SQuAD] Processed 200/200

B: NQ -> NQ...
  [Dense-RAG | NQ->NQ] Processed 50/200
  [Dense-RAG | NQ->NQ] Processed 100/200
  [Dense-RAG | NQ->NQ] Processed 150/200
  [Dense-RAG | NQ->NQ] Processed 200/200

C: TriviaQA -> TriviaQA...
  [Dense-RAG | Trivia->Trivia] Processed 50/200
  [Dense-RAG | Trivia->Trivia] Processed 100/200
  [Dense-RAG | Trivia->Trivia] Processed 150/200
  [Dense-RAG | Trivia->Trivia] Processed 200/200

D: SQuAD -> NQ...
  [Dense-RAG | SQuAD->NQ] Processed 50/200
  [Dense-RAG | SQuAD->NQ] Processed 100/200
  [Dense-RAG | SQuAD->NQ] Processed 150/200
  [Dense-RAG | SQuAD->NQ] Processed 200/200

E: SQuAD -> TriviaQA...
  [Dense-RAG | SQuAD->Trivia] Processed 50/200
  [Dense-RAG | SQuAD->Trivia] Processed 100/200
  [Dense-RAG | SQuAD->Trivia] Processed 150/20

---
## Section 12 — Full Scenario Comparison Table

All components and all scenarios consolidated. This is the master results table for the report.

In [ ]:
print('=== FULL SCENARIO COMPARISON TABLE ===')

rows = [
    {'Scenario': 'A: SQuAD->SQuAD',    'Type': 'in-domain',    'LLM-Only F1': llm_results['squad']['f1'],  'BM25 Recall@K': bm25_results['A_squad_squad']['recall@k'],   'Dense Recall@K': dense_results['A_squad_squad']['recall@k'],   'BM25-RAG F1': rag_bm25['A']['f1'],  'Dense-RAG F1': rag_dense['A']['f1']},
    {'Scenario': 'B: NQ->NQ',          'Type': 'in-domain',    'LLM-Only F1': llm_results['nq']['f1'],     'BM25 Recall@K': bm25_results['B_nq_nq']['recall@k'],          'Dense Recall@K': dense_results['B_nq_nq']['recall@k'],          'BM25-RAG F1': rag_bm25['B']['f1'],  'Dense-RAG F1': rag_dense['B']['f1']},
    {'Scenario': 'C: Trivia->Trivia',  'Type': 'in-domain',    'LLM-Only F1': llm_results['trivia']['f1'], 'BM25 Recall@K': bm25_results['C_trivia_trivia']['recall@k'],  'Dense Recall@K': dense_results['C_trivia_trivia']['recall@k'],  'BM25-RAG F1': rag_bm25['C']['f1'],  'Dense-RAG F1': rag_dense['C']['f1']},
    {'Scenario': 'D: SQuAD->NQ',       'Type': 'cross-domain', 'LLM-Only F1': llm_results['squad']['f1'],  'BM25 Recall@K': bm25_results['D_squad_nq']['recall@k'],       'Dense Recall@K': dense_results['D_squad_nq']['recall@k'],       'BM25-RAG F1': rag_bm25['D']['f1'],  'Dense-RAG F1': rag_dense['D']['f1']},
    {'Scenario': 'E: SQuAD->Trivia',   'Type': 'cross-domain', 'LLM-Only F1': llm_results['squad']['f1'],  'BM25 Recall@K': bm25_results['E_squad_trivia']['recall@k'],   'Dense Recall@K': dense_results['E_squad_trivia']['recall@k'],   'BM25-RAG F1': rag_bm25['E']['f1'],  'Dense-RAG F1': rag_dense['E']['f1']},
    {'Scenario': 'F: Trivia->SQuAD',   'Type': 'cross-domain', 'LLM-Only F1': llm_results['trivia']['f1'], 'BM25 Recall@K': bm25_results['F_trivia_squad']['recall@k'],   'Dense Recall@K': dense_results['F_trivia_squad']['recall@k'],   'BM25-RAG F1': rag_bm25['F']['f1'],  'Dense-RAG F1': rag_dense['F']['f1']},
]

full_df = pd.DataFrame(rows)
print(full_df.to_string(index=False))
full_df.to_csv('rag_full_results.csv', index=False)
print('\nSaved to rag_full_results.csv')

=== FULL SCENARIO COMPARISON TABLE ===
         Scenario         Type  LLM-Only F1  BM25 Recall@K  Dense Recall@K  BM25-RAG F1  Dense-RAG F1
  A: SQuAD->SQuAD    in-domain       0.0278          0.750           0.925       0.1178        0.1256
        B: NQ->NQ    in-domain       0.0404          0.015           0.015       0.0374        0.0409
C: Trivia->Trivia    in-domain       0.0401          0.255           0.325       0.0660        0.0595
     D: SQuAD->NQ cross-domain       0.0278          0.015           0.005       0.0355        0.0295
 E: SQuAD->Trivia cross-domain       0.0278          0.055           0.050       0.0342        0.0264
 F: Trivia->SQuAD cross-domain       0.0401          0.025           0.070       0.0321        0.0324

Saved to rag_full_results.csv


---
## Section 13 — Full Pipeline Spot-Check

For each scenario we print N_SPOT_CHECK complete examples showing the full pipeline:
- The question and ground truth
- Top retrieved chunk for BM25 and Dense (with hit/miss)
- LLM-only prediction
- BM25-RAG prediction
- Dense-RAG prediction
- F1 scores for each

This is the qualitative analysis that shows **where failures happen** — retrieval, generation, or both.

In [ ]:
def full_pipeline_spotcheck(data, bm25_fn, dense_fn, label, n=N_SPOT_CHECK):
    """Print complete pipeline trace for N examples."""
    print(f'\n{"="*70}')
    print(f'SPOT-CHECK: {label}')
    print('='*70)

    for i, sample in enumerate(data[:n]):
        q = sample['question']
        a = sample['answer']

        bm25_res    = bm25_fn(q)
        dense_res   = dense_fn(q)
        bm25_chunks = [r[0] for r in bm25_res]
        dense_chunks= [r[0] for r in dense_res]

        llm_pred   = generate_answer(build_llm_only_prompt(q))
        bm25_pred  = generate_answer(build_rag_prompt(bm25_chunks[:K_GENERATE], q))
        dense_pred = generate_answer(build_rag_prompt(dense_chunks[:K_GENERATE], q))

        bm25_hit  = '✅' if recall_at_k(a, bm25_chunks) else '❌'
        dense_hit = '✅' if recall_at_k(a, dense_chunks) else '❌'

        print(f'\nExample {i+1}')
        print(f'  QUESTION:       {q[:150]}')
        print(f'  GROUND TRUTH:   {a[:150]}')
        print(f'  ─── Retrieval ───')
        print(f'  BM25  [{bm25_hit}]: {bm25_chunks[0][:150] if bm25_chunks else "(none)"}...')
        print(f'  Dense [{dense_hit}]: {dense_chunks[0][:150] if dense_chunks else "(none)"}...')
        print(f'  ─── Generation ───')
        print(f'  LLM-only:   {llm_pred[:150]}')
        print(f'  BM25-RAG:   {bm25_pred[:150]}')
        print(f'  Dense-RAG:  {dense_pred[:150]}')
        print(f'  ─── Scores ───')
        print(f'  LLM-only   F1={f1_score(llm_pred, a):.3f}  EM={exact_match(llm_pred, a)}')
        print(f'  BM25-RAG   F1={f1_score(bm25_pred, a):.3f}  EM={exact_match(bm25_pred, a)}')
        print(f'  Dense-RAG  F1={f1_score(dense_pred, a):.3f}  EM={exact_match(dense_pred, a)}')

print('Spot-check function defined.')

Spot-check function defined.


In [ ]:
# Scenario A: SQuAD In-Domain
full_pipeline_spotcheck(
    squad_data,
    lambda q: retrieve_bm25(q, bm25_squad, squad_chunks),
    lambda q: retrieve_dense(q, faiss_squad, squad_chunks),
    'Scenario A: SQuAD -> SQuAD (in-domain)'
)


SPOT-CHECK: Scenario A: SQuAD -> SQuAD (in-domain)

Example 1
  QUESTION:       To whom did the Virgin Mary allegedly appear in 1858 in Lourdes France?
  GROUND TRUTH:   Saint Bernadette Soubirous
  ─── Retrieval ───
  BM25  [✅]: Architecturally, the school has a Catholic character. Atop the Main Building's gold dome is a golden statue of the Virgin Mary. Immediately in front o...
  Dense [✅]: Architecturally, the school has a Catholic character. Atop the Main Building's gold dome is a golden statue of the Virgin Mary. Immediately in front o...
  ─── Generation ───
  LLM-only:   The Virgin Mary allegedly appeared to a group of people in Lourdes, France in 1858. This event is known as the "Lourdes Miracle" or "Miracle of Lourde
  BM25-RAG:   The Virgin Mary allegedly appeared to Saint Bernadette Soubirous in 1858 in Lourdes, France.
  Dense-RAG:  The Virgin Mary allegedly appeared to Saint Bernadette Soubirous in 1858 in Lourdes, France. This event is described in detail in the text pr

In [ ]:
# Scenario B: NQ In-Domain
full_pipeline_spotcheck(
    nq_data,
    lambda q: retrieve_bm25(q, bm25_nq, nq_chunks),
    lambda q: retrieve_dense(q, faiss_nq, nq_chunks),
    'Scenario B: NQ -> NQ (in-domain)'
)


SPOT-CHECK: Scenario B: NQ -> NQ (in-domain)

Example 1
  QUESTION:       when is the last episode of season 8 of the walking dead
  GROUND TRUTH:   March 18, 2018
  ─── Retrieval ───
  BM25  [❌]: List of proxy wars - Wikipedia List of proxy wars From Wikipedia, the free encyclopedia Jump to navigation Jump to search This article has multiple is...
  Dense [❌]: Ghost Whisperer (season 4) - Wikipedia .mw-editfont-monospace .mw-editfont-sans-serif .mw-editfont-serif .mw-editfont-monospace,.mw-editfont-sans-seri...
  ─── Generation ───
  LLM-only:   The last episode of Season 8 of Walking Dead aired on October 15, 2016. This was the final episode of the series before its cancellation in 2017. It c
  BM25-RAG:   I'm sorry, but I cannot provide an answer without knowing what specific question you're asking. Could you please clarify your request? Are you looking
  Dense-RAG:  The last episode of Season 8 of Walking Dead was "The Last Stand". This episode aired on January 25, 2016. It conclu

In [ ]:
# Scenario C: TriviaQA In-Domain
full_pipeline_spotcheck(
    trivia_data,
    lambda q: retrieve_bm25(q, bm25_trivia, trivia_chunks),
    lambda q: retrieve_dense(q, faiss_trivia, trivia_chunks),
    'Scenario C: TriviaQA -> TriviaQA (in-domain)'
)


SPOT-CHECK: Scenario C: TriviaQA -> TriviaQA (in-domain)

Example 1
  QUESTION:       Which American-born Sinclair won the Nobel Prize for Literature in 1930?
  GROUND TRUTH:   Sinclair Lewis
  ─── Retrieval ───
  BM25  [✅]: An American Nobel Prize in Literature - Nov 05, 1930 - HISTORY.com
An American Nobel Prize in Literature
Share this:
An American Nobel Prize in Litera...
  Dense [✅]: An American Nobel Prize in Literature - Nov 05, 1930 - HISTORY.com
An American Nobel Prize in Literature
Share this:
An American Nobel Prize in Litera...
  ─── Generation ───
  LLM-only:   Sinclair Lewis won the Nobel Prize for Literature in 1930. He was born on January 2, 1885, in Chicago, Illinois, United States. Lewis is best known fo
  BM25-RAG:   Sinclair Lewis won the Nobel Prize in Literature in 1930. He was born in Sauk Center, Minnesota, and established his literary reputation in the 1920s 
  Dense-RAG:  The American-born Sinclair Lewis won the Nobel Prize in Literature in 1930. According to

In [ ]:
# Scenario E: SQuAD -> TriviaQA (Cross-Domain)
full_pipeline_spotcheck(
    squad_data,
    lambda q: retrieve_bm25(q, bm25_trivia, trivia_chunks),
    lambda q: retrieve_dense(q, faiss_trivia, trivia_chunks),
    'Scenario E: SQuAD -> TriviaQA (cross-domain)'
)


SPOT-CHECK: Scenario E: SQuAD -> TriviaQA (cross-domain)

Example 1
  QUESTION:       To whom did the Virgin Mary allegedly appear in 1858 in Lourdes France?
  GROUND TRUTH:   Saint Bernadette Soubirous
  ─── Retrieval ───
  BM25  [❌]: CARLOS LEÓN, Actor, Father of Madonna’s child “Lola”. (Born: Havana) *** Carlos León, Actor, Padre de la hija de Madonna “Lola”. (Nacido en La Habana)...
  Dense [❌]: of Lourdes. Father with Madonna of Lourdes Maria Ciccone Leon (Lourdes Leon), born 14 October 1996. Lourdes comes from the Marian shrine in France whi...
  ─── Generation ───
  LLM-only:   The Virgin Mary allegedly appeared to a group of people in Lourdes, France in 1858. This event is known as the "Lourdes Miracle" or "Miracle of Lourde
  BM25-RAG:   The Virgin Mary allegedly appeared in Lourdes France in 1858. According to the passage, "Madonna's daughter Lourdes Is The Spitting Image Of Her Mum A
  Dense-RAG:  The Virgin Mary allegedly appeared in Lourdes France in 1858. According to th

In [ ]:
# Scenario F: TriviaQA -> SQuAD (Cross-Domain)
full_pipeline_spotcheck(
    trivia_data,
    lambda q: retrieve_bm25(q, bm25_squad, squad_chunks),
    lambda q: retrieve_dense(q, faiss_squad, squad_chunks),
    'Scenario F: TriviaQA -> SQuAD (cross-domain)'
)


SPOT-CHECK: Scenario F: TriviaQA -> SQuAD (cross-domain)

Example 1
  QUESTION:       Which American-born Sinclair won the Nobel Prize for Literature in 1930?
  GROUND TRUTH:   Sinclair Lewis
  ─── Retrieval ───
  BM25  [❌]: The first sulfonamide and first commercially available antibacterial, Prontosil, was developed by a research team led by Gerhard Domagk in 1932 at the...
  Dense [❌]: The rise of Hitler and other dictators in the 1930s forced numerous Catholic intellectuals to flee Europe; president John O'Hara brought many to Notre...
  ─── Generation ───
  LLM-only:   Sinclair Lewis won the Nobel Prize for Literature in 1930. He was born on January 2, 1885, in Chicago, Illinois, United States. Lewis is best known fo
  BM25-RAG:   The American-born poet and novelist Sinclair Lewis won the Nobel Prize for Literature in 1930. He was born on January 27, 1885, in Chicago, Illinois, 
  Dense-RAG:  Ivan Meštrović won the Nobel Prize for Literature in 1930. He was a renowned sculptor wh

---
## Section 14 — Failure Analysis

Each example is classified into one of three categories:
1. **Retrieval failure** — correct chunk not retrieved (Recall@K = 0)
2. **Generation failure** — correct chunk retrieved but model answered incorrectly (F1 < 0.3)
3. **Success** — retrieval hit and F1 >= 0.3

This directly answers Research Question 3: *Where does the RAG pipeline fail?*

In [ ]:
def classify_failures(data, retriever_fn, label, n=N_SAMPLES):
    """Classify each sample as retrieval failure, generation failure, or success."""
    ret_fail, gen_fail, success = 0, 0, 0
    examples = {'retrieval': [], 'generation': [], 'success': []}

    for i, sample in enumerate(data[:n]):
        q = sample['question']
        a = sample['answer']

        results = retriever_fn(q)
        chunks  = [r[0] for r in results]
        r       = recall_at_k(a, chunks)
        pred    = generate_answer(build_rag_prompt(chunks[:K_GENERATE], q))
        f1      = f1_score(pred, a)

        entry = {'question': q, 'answer': a, 'prediction': pred,
                 'top_chunk': chunks[0] if chunks else '', 'f1': round(f1, 3)}

        if not r:
            ret_fail += 1
            if len(examples['retrieval']) < 2:
                examples['retrieval'].append(entry)
        elif f1 < 0.3:
            gen_fail += 1
            if len(examples['generation']) < 2:
                examples['generation'].append(entry)
        else:
            success += 1
            if len(examples['success']) < 2:
                examples['success'].append(entry)

        if (i+1) % 50 == 0:
            print(f'  [{label}] Processed {i+1}/{n}')

    total = ret_fail + gen_fail + success
    print(f'\n  {label}')
    print(f'  Retrieval failures:  {ret_fail}/{total} ({100*ret_fail/total:.1f}%)')
    print(f'  Generation failures: {gen_fail}/{total} ({100*gen_fail/total:.1f}%)')
    print(f'  Successes:           {success}/{total} ({100*success/total:.1f}%)')

    return {
        'label':               label,
        'retrieval_fail_pct':  round(100*ret_fail/total, 1),
        'generation_fail_pct': round(100*gen_fail/total, 1),
        'success_pct':         round(100*success/total, 1)
    }, examples

print('Failure analysis function defined.')

Failure analysis function defined.


In [ ]:
print('=== FAILURE ANALYSIS ===')
failure_results = {}

print('\nAnalyzing Dense RAG failures across scenarios...')

failure_results['A_squad'], ex_A = classify_failures(
    squad_data, lambda q: retrieve_dense(q, faiss_squad, squad_chunks), 'Dense | SQuAD->SQuAD')

failure_results['B_nq'], ex_B = classify_failures(
    nq_data, lambda q: retrieve_dense(q, faiss_nq, nq_chunks), 'Dense | NQ->NQ')

failure_results['C_trivia'], ex_C = classify_failures(
    trivia_data, lambda q: retrieve_dense(q, faiss_trivia, trivia_chunks), 'Dense | Trivia->Trivia')

failure_results['E_cross'], ex_E = classify_failures(
    squad_data, lambda q: retrieve_dense(q, faiss_trivia, trivia_chunks), 'Dense | SQuAD->Trivia')

failure_results['F_cross'], ex_F = classify_failures(
    trivia_data, lambda q: retrieve_dense(q, faiss_squad, squad_chunks), 'Dense | Trivia->SQuAD')

print('\n=== FAILURE ANALYSIS SUMMARY ===')
print(pd.DataFrame(failure_results).T.to_string())

=== FAILURE ANALYSIS ===

Analyzing Dense RAG failures across scenarios...
  [Dense | SQuAD->SQuAD] Processed 50/200
  [Dense | SQuAD->SQuAD] Processed 100/200
  [Dense | SQuAD->SQuAD] Processed 150/200
  [Dense | SQuAD->SQuAD] Processed 200/200

  Dense | SQuAD->SQuAD
  Retrieval failures:  15/200 (7.5%)
  Generation failures: 172/200 (86.0%)
  Successes:           13/200 (6.5%)
  [Dense | NQ->NQ] Processed 50/200
  [Dense | NQ->NQ] Processed 100/200
  [Dense | NQ->NQ] Processed 150/200
  [Dense | NQ->NQ] Processed 200/200

  Dense | NQ->NQ
  Retrieval failures:  197/200 (98.5%)
  Generation failures: 3/200 (1.5%)
  Successes:           0/200 (0.0%)
  [Dense | Trivia->Trivia] Processed 50/200
  [Dense | Trivia->Trivia] Processed 100/200
  [Dense | Trivia->Trivia] Processed 150/200
  [Dense | Trivia->Trivia] Processed 200/200

  Dense | Trivia->Trivia
  Retrieval failures:  135/200 (67.5%)
  Generation failures: 59/200 (29.5%)
  Successes:           6/200 (3.0%)
  [Dense | SQuAD->Trivi

In [ ]:
# ── Failure Examples ──────────────────────────────────────────────
print('=== FAILURE EXAMPLES ===')

def print_failure_examples(examples, label):
    print(f'\n--- {label} ---')
    for ftype, exlist in examples.items():
        if exlist:
            print(f'\n  [{ftype.upper()} EXAMPLES]')
            for ex in exlist:
                print(f'    Q:      {ex["question"][:120]}')
                print(f'    Truth:  {ex["answer"][:100]}')
                print(f'    Chunk:  {ex["top_chunk"][:120]}...')
                print(f'    Pred:   {ex["prediction"][:120]}')
                print(f'    F1:     {ex["f1"]}')
                print()

print_failure_examples(ex_A, 'SQuAD In-Domain')
print_failure_examples(ex_B, 'NQ In-Domain')
print_failure_examples(ex_C, 'TriviaQA In-Domain')
print_failure_examples(ex_E, 'SQuAD->TriviaQA Cross-Domain')
print_failure_examples(ex_F, 'TriviaQA->SQuAD Cross-Domain')

=== FAILURE EXAMPLES ===

--- SQuAD In-Domain ---

  [RETRIEVAL EXAMPLES]
    Q:      In what year was a Master of Arts course first offered at Notre Dame?
    Truth:  1854
    Chunk:  All of Notre Dame's undergraduate students are a part of one of the five undergraduate colleges at the school or are in ...
    Pred:   The Master of Arts (MA) course was first offered at Notre Dame in the 1854–1855 academic year. This occurred in the earl
    F1:     0.0

    Q:      What is the title of Notre Dame's Theodore Hesburgh?
    Truth:  President Emeritus of the University of Notre Dame
    Chunk:  The library system of the university is divided between the main library and each of the colleges and schools. The main ...
    Pred:   Theodore Hesburgh was the president of Notre Dame University for 35 years, during which the university underwent signifi
    F1:     0.245


  [GENERATION EXAMPLES]
    Q:      To whom did the Virgin Mary allegedly appear in 1858 in Lourdes France?
    Truth:  Sain

---
## Section 15 — Save All Results

In [ ]:
all_results = {
    'bm25_retrieval':   bm25_results,
    'dense_retrieval':  dense_results,
    'llm_only':         llm_results,
    'bm25_rag':         rag_bm25,
    'dense_rag':        rag_dense,
    'failure_analysis': failure_results
}

with open('rag_all_results.json', 'w') as f:
    json.dump(all_results, f, indent=2)

full_df.to_csv('rag_full_comparison.csv', index=False)

print('All results saved to Google Drive:')
print('  rag_all_results.json')
print('  rag_full_comparison.csv')

All results saved to Google Drive:
  rag_all_results.json
  rag_full_comparison.csv
